# Task A -- Task-adaptive pretraining before demojized MuRIL fine-tuning

This experiment compares stock demojized MuRIL with MuRIL adapted using masked-language-model training on Task A text before the same binary classifier fine-tuning.

| setting | value |
|---|---|
| control | stock google/muril-base-cased |
| variant | Task-A-domain TAPT, then demojized MuRIL fine-tuning |
| TAPT text | only the 85% classifier-training side of binary_train.csv plus OffensEval Kannada |
| classifier evaluation | fixed 15% stratified holdout, split seed 42 |
| classifier | two-layer reinitialization, FGM, EMA, six epochs |
| model seed | 42 for both arms |

The holdout comments, Task A validation inputs, and Task B files are excluded from TAPT. Both arms produce validated Task A submission ZIPs. Expected runtime is roughly 2--4 hours on a T4 or RTX 3070.

In [ ]:
import os, pathlib, re, shutil, subprocess, sys
import numpy as np
import pandas as pd

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", "task-b", "--depth", "1",
                    "https://github.com/robinpnalex/Hastika-ICON2026.git", WORK], check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
sys.path.insert(0, os.path.join(WORK, "src"))
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"', shell=True, check=True)
import torch
assert torch.cuda.is_available(), "no GPU -- select a CUDA-enabled runtime"
print("gpu:", torch.cuda.get_device_name(0))

def run(cmd, log=None):
    print("$", " ".join(cmd), flush=True)
    fh = open(log, "w") if log else None
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        sys.stdout.write(line)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}: {cmd}")


## 1. Create the legal TAPT corpus

The split is recreated exactly as the Task A classifier uses it: deduplicate first, then take a stratified 15% holdout with split seed 42. Only the 85% training-side comments are written to the temporary TAPT CSV.

In [ ]:
from sklearn.model_selection import train_test_split
from hastika.common.preprocessing import dedupe_index

raw = pd.read_csv("data/raw/binary_train.csv")
keep = dedupe_index(raw["Comment"].tolist(), raw["Label"].tolist(), "task A")
train = raw.iloc[keep].reset_index(drop=True)
y = (train["Label"] == "Hate").astype(int).to_numpy()
tr_i, va_i = train_test_split(np.arange(len(train)), test_size=0.15, stratify=y, random_state=42)
TAPT_INPUT = "/kaggle/working/task_a_tapt_train.csv"
train.iloc[tr_i][["Comment"]].to_csv(TAPT_INPUT, index=False)
print(f"deduplicated rows: {len(train)}")
print(f"classifier training rows: {len(tr_i)}; holdout rows: {len(va_i)}")
assert set(tr_i).isdisjoint(set(va_i))
assert len(pd.read_csv(TAPT_INPUT)) == len(tr_i)


## 2. Run Task-A-domain TAPT

The repository MLM implementation is called with an explicit custom corpus. It reads only the temporary Task A comments and permitted OffensEval text; labels are never read.

In [ ]:
TAPT_OUT = "artifacts/runs/tapt-task-a-train85"
TAPT_LOG = "artifacts/logs/tapt_task_a_train85.log"
if (pathlib.Path(TAPT_OUT) / "config.json").exists():
    print("using existing TAPT checkpoint:", TAPT_OUT)
else:
    run([sys.executable, "-u", "-m", "hastika.task_b.tapt",
         "--model", "google/muril-base-cased", "--corpus", TAPT_INPUT,
         "data/external/offenseval_kn.csv", "--epochs", "8", "--val-frac", "0.05",
         "--min-words", "1", "--out", TAPT_OUT], log=TAPT_LOG)
assert pathlib.Path(TAPT_OUT).is_dir()
if pathlib.Path(TAPT_LOG).exists():
    t = pathlib.Path(TAPT_LOG).read_text()
    m = re.search(r"MLM trains on (\d+) of (\d+) comments, (\d+) held out", t)
    assert m, "TAPT corpus count not found"
    used, total, held = map(int, m.groups())
    print(f"TAPT trains on {used} of {total}; {held} held out for MLM perplexity")
    assert used + held == total and held > 0
print("TAPT checkpoint ready:", TAPT_OUT)


## 3. Compare stock and TAPT MuRIL

Both arms use the same holdout, seed, demojization, optimizer settings, and final-checkpoint selection. Batch 8 with accumulation 2 preserves an effective batch size of 16 while reducing 3070 memory pressure.

In [ ]:
COMMON = ["--folds", "0", "--seeds", "42", "--epochs", "6", "--bs", "8", "--grad-accum", "2", "--eval-bs", "32", "--select", "last", "--reinit-layers", "2"]
ARMS = [("task_a_muril_control", ["--model", "google/muril-base-cased"]),
        ("task_a_muril_tapt", ["--model", TAPT_OUT])]
for tag, model_args in ARMS:
    run([sys.executable, "-u", "-m", "hastika.models.muril", "--tag", tag,
         *model_args, *COMMON], log=f"artifacts/logs/{tag}.log")
print("both classifier arms completed")


## 4. Score and package validation predictions

The two ZIPs contain one bare predictions.csv with the id,label schema. Use the holdout scores to decide whether a full-data TAPT fit is worth submitting against the current 0.8163 MuRIL result.

In [ ]:
records = []
for tag, _ in ARMS:
    text = pathlib.Path(f"artifacts/logs/{tag}.log").read_text()
    scores = re.findall(r"holdout macro-F1 ([0-9.]+)", text)
    assert scores, f"no holdout score found for {tag}"
    records.append({"run": tag, "last_macro_f1": float(scores[0]),
                    "best_macro_f1": float(scores[1]) if len(scores) > 1 else None})
summary = pd.DataFrame(records)
display(summary.round(4))
delta = summary.loc[summary.run == "task_a_muril_tapt", "last_macro_f1"].iloc[0] - summary.loc[summary.run == "task_a_muril_control", "last_macro_f1"].iloc[0]
print(f"TAPT minus stock MuRIL: {delta:+.4f} holdout macro-F1")
for tag, _ in ARMS:
    run([sys.executable, "-m", "hastika.common.submission", "--task", "a",
         "--pred", f"artifacts/runs/{tag}/predictions.csv",
         "--out", f"/kaggle/working/{tag}.zip"])
print("Download task_a_muril_control.zip and task_a_muril_tapt.zip from the Kaggle Output tab.")


## 5. Copy outputs

Download this folder from Kaggle for the experiment record.

In [ ]:
OUT = pathlib.Path("/kaggle/working/task_a_tapt_outputs")
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "task_a_tapt_summary.csv", index=False)
if pathlib.Path(TAPT_LOG).exists():
    shutil.copy2(TAPT_LOG, OUT / pathlib.Path(TAPT_LOG).name)
for tag, _ in ARMS:
    shutil.copy2(f"artifacts/logs/{tag}.log", OUT / f"{tag}.log")
    shutil.copy2(f"artifacts/runs/{tag}/holdout_probs.npy", OUT / f"{tag}_holdout_probs.npy")
    shutil.copy2(f"/kaggle/working/{tag}.zip", OUT / f"{tag}.zip")
    shutil.copy2(f"artifacts/runs/{tag}/predictions.csv", OUT / f"{tag}_predictions.csv")
print("download task_a_tapt_outputs/ from the Kaggle Output tab")
